# DOTA: Scale-Aware Automatic Augmentation vs Adaptive Rule Policy

???? ??????? ????????? **??????? ?????????** ???? ???????? ?? ????? ? ??? ?? ???????? DOTA:

1. `Scale-Aware Automatic Augmentation` (???????????? budget-search ?? ?????????).
2. `??? ??????` (rule-based adaptive policy ?? `src/policy/rule_engine.py`).

????????? ??????????? ? 3 ?????:
- ?????? ???????? ? ?????????? adaptive policy,
- ???????? scale-aware ????? ?????????? (???????? ??????????),
- ????????? ?????????? ? COCOeval-??????? (??????? `AP_small`, `AP_tiny`).


In [ ]:
# Step 0: install dependencies in Colab runtime.
%pip install -q ultralytics pycocotools albumentations pyyaml tqdm pandas


In [ ]:
# Step 1: mount Google Drive and define paths.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = Path('/content/small_objects_auto_aug')
DRIVE_ROOT = Path('/content/drive/MyDrive')

DATASET_ROOT = DRIVE_ROOT / 'datasets' / 'dota_yolo'
assert DATASET_ROOT.exists(), f'Dataset root not found: {DATASET_ROOT}'

EXPERIMENT_ROOT = DRIVE_ROOT / 'experiments' / 'small_objects_auto_aug' / 'dota_scaleaware_vs_adaptive'
RUNS_ROOT = EXPERIMENT_ROOT / 'runs'
ARTIFACTS_ROOT = EXPERIMENT_ROOT / 'artifacts'

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

print('DATASET_ROOT =', DATASET_ROOT)
print('EXPERIMENT_ROOT =', EXPERIMENT_ROOT)
print('RUNS_ROOT =', RUNS_ROOT)
print('ARTIFACTS_ROOT =', ARTIFACTS_ROOT)


In [ ]:
# Step 2: clone/open project repository in runtime.
import subprocess
import sys

REPO_URL = 'https://github.com/s44w/small_objects_auto_aug.git'
if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Using PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
# Step 3: build runtime config for this experiment.
import yaml

base_cfg_path = PROJECT_ROOT / 'configs' / 'dota_config.yaml'
runtime_cfg_path = PROJECT_ROOT / 'artifacts' / 'dota_scaleaware_vs_adaptive_runtime.yaml'
runtime_cfg_path.parent.mkdir(parents=True, exist_ok=True)

with base_cfg_path.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['kind'] = 'yolo_generic'
cfg['dataset']['mode'] = 'manual'
cfg['dataset']['root'] = str(DATASET_ROOT)
cfg['training']['data_yaml'] = str(DATASET_ROOT / 'dataset.yaml')
cfg['training']['project_dir'] = str(RUNS_ROOT)
cfg['training']['run_ablation'] = False
cfg['training']['cache'] = 'disk'
cfg['training']['val_during_train'] = True

with runtime_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

print('runtime_cfg_path:', runtime_cfg_path)
print('training.project_dir:', cfg['training']['project_dir'])


In [ ]:
# Step 4: dataset analysis + adaptive policy generation.
from src.utils.io import load_yaml
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts

runtime_cfg = load_yaml(runtime_cfg_path)

stats_dir = ARTIFACTS_ROOT / 'stats'
policy_dir = ARTIFACTS_ROOT / 'policy'
stats_dir.mkdir(parents=True, exist_ok=True)
policy_dir.mkdir(parents=True, exist_ok=True)

analyzer_cfg = DatasetAnalyzerConfig(
    small_max_area=float(runtime_cfg['analysis']['small_max_area']),
    medium_max_area=float(runtime_cfg['analysis']['medium_max_area']),
    tiny_max_area=float(runtime_cfg['analysis']['tiny_max_area']),
    image_extensions=tuple(runtime_cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
    generate_plots=False,
    show_progress=True,
)

stats = analyze_dataset(
    dataset_root=DATASET_ROOT,
    output_dir=stats_dir,
    splits=tuple(runtime_cfg['dataset'].get('splits', ['train', 'val'])),
    config=analyzer_cfg,
)

rule_cfg = RuleEngineConfig.from_project_config(runtime_cfg)
adaptive_policy, adaptive_report = generate_policy_from_stats(stats, cfg=rule_cfg)
adaptive_paths = save_policy_artifacts(adaptive_policy, adaptive_report, output_dir=policy_dir)

print('Stats:', stats_dir / 'dataset_stats.json')
print('Adaptive policy JSON:', adaptive_paths['policy_json'])
print('Adaptive decision report:', adaptive_paths['decision_report'])


In [ ]:
# Step 5: build Scale-Aware AutoAug candidate pool from dataset stats.
import random
import copy
from src.policy.policy_schema import filter_ultralytics_detect_args
from src.utils.io import dump_json

SEED = int(runtime_cfg['project'].get('seed', 42))
NUM_SCALE_AWARE_CANDIDATES = 8

def build_scale_aware_candidates(dataset_stats: dict, num_candidates: int = 8, seed: int = 42):
    ratios = dataset_stats['splits']['train']['ratios']
    density = dataset_stats['splits']['train']['density']

    small_ratio = float(ratios.get('small_ratio', 0.0))
    tiny_ratio = float(ratios.get('tiny_ratio', 0.0))
    opi_mean = float(density['objects_per_image'].get('mean', 0.0))
    opm_mean = float(density['objects_per_mpix'].get('mean', 0.0))

    is_small_heavy = small_ratio >= 0.55 or tiny_ratio >= 0.20
    is_dense = opi_mean >= 15.0 or opm_mean >= 30.0

    # Scale-aware ??????? ????????:
    # - ??? small/tiny ???????? ???????????? ??????????? ?????????
    # - ??? dense ???? ????????? ????????? mosaic
    base = {
        'mosaic': 0.7 if is_dense else 0.5,
        'close_mosaic': 12,
        'hsv_h': 0.015,
        'hsv_s': 0.55,
        'hsv_v': 0.45,
        'degrees': 3.0 if is_small_heavy else 5.0,
        'translate': 0.06 if is_small_heavy else 0.10,
        'scale': 0.35 if is_small_heavy else 0.50,
        'perspective': 0.0,
        'fliplr': 0.5,
        'flipud': 0.0,
        'mixup': 0.0,
        'cutmix': 0.0,
    }

    rng = random.Random(seed)

    mosaic_vals = [max(0.0, min(1.0, base['mosaic'] + d)) for d in (-0.2, -0.1, 0.0, 0.1, 0.2)]
    scale_vals = [max(0.15, min(0.8, base['scale'] + d)) for d in (-0.12, -0.07, 0.0, 0.07, 0.12)]
    translate_vals = [max(0.02, min(0.18, base['translate'] + d)) for d in (-0.03, -0.01, 0.0, 0.01, 0.03)]
    degrees_vals = [max(0.0, min(10.0, base['degrees'] + d)) for d in (-2.0, -1.0, 0.0, 1.0, 2.0)]
    hsv_s_vals = [max(0.2, min(0.9, base['hsv_s'] + d)) for d in (-0.2, -0.1, 0.0, 0.1)]
    hsv_v_vals = [max(0.2, min(0.7, base['hsv_v'] + d)) for d in (-0.15, -0.05, 0.0, 0.05)]

    candidates = []
    for idx in range(num_candidates):
        args = copy.deepcopy(base)
        args.update({
            'mosaic': rng.choice(mosaic_vals),
            'scale': rng.choice(scale_vals),
            'translate': rng.choice(translate_vals),
            'degrees': rng.choice(degrees_vals),
            'hsv_s': rng.choice(hsv_s_vals),
            'hsv_v': rng.choice(hsv_v_vals),
        })

        if tiny_ratio > 0.30 and idx % 3 == 0:
            args['mosaic'] = max(0.2, args['mosaic'] - 0.2)

        candidates.append({
            'candidate_id': f'scaleaware_{idx:02d}',
            'ultralytics_args': filter_ultralytics_detect_args(args),
            'meta': {
                'is_small_heavy': is_small_heavy,
                'is_dense': is_dense,
                'small_ratio': small_ratio,
                'tiny_ratio': tiny_ratio,
                'objects_per_image_mean': opi_mean,
                'objects_per_mpix_mean': opm_mean,
            }
        })

    return candidates

scaleaware_candidates = build_scale_aware_candidates(
    dataset_stats=stats,
    num_candidates=NUM_SCALE_AWARE_CANDIDATES,
    seed=SEED,
)

cand_dir = ARTIFACTS_ROOT / 'scaleaware_candidates'
cand_dir.mkdir(parents=True, exist_ok=True)
dump_json(scaleaware_candidates, cand_dir / 'scaleaware_candidates.json')

print('Saved candidates:', cand_dir / 'scaleaware_candidates.json')
print('First candidate:', scaleaware_candidates[0])


In [ ]:
# Step 6: pilot search over Scale-Aware candidates (short budget).
import pandas as pd
from pathlib import Path
from src.training.train_runner import TrainRunConfig, run_train_mode
from src.utils.io import dump_json, load_json

RUN_SCALE_AWARE_SEARCH = False
FORCE_RERUN_SEARCH = False

PILOT_EPOCHS = 6
PILOT_IMGSZ = 960
PILOT_BATCH = 8
PILOT_FRACTION = 0.30

search_artifacts_dir = ARTIFACTS_ROOT / 'scaleaware_search'
search_artifacts_dir.mkdir(parents=True, exist_ok=True)
search_leaderboard_path = search_artifacts_dir / 'leaderboard.json'

def _best_map5095_from_results_csv(results_csv: Path) -> float:
    if not results_csv.exists():
        return -1.0
    try:
        df = pd.read_csv(results_csv)
    except Exception:
        return -1.0
    col = 'metrics/mAP50-95(B)'
    if col not in df.columns:
        return -1.0
    return float(df[col].max())

if RUN_SCALE_AWARE_SEARCH or (not search_leaderboard_path.exists()) or FORCE_RERUN_SEARCH:
    pilot_cfg = TrainRunConfig(
        data_yaml=str(DATASET_ROOT / 'dataset.yaml'),
        model=str(runtime_cfg['training']['model']),
        epochs=int(PILOT_EPOCHS),
        imgsz=int(PILOT_IMGSZ),
        batch=int(PILOT_BATCH),
        device=0,
        workers=int(runtime_cfg['training'].get('workers', 2)),
        fraction=float(PILOT_FRACTION),
        project_dir=str(RUNS_ROOT / 'scaleaware_search'),
        seed=int(SEED),
        deterministic=bool(runtime_cfg['project'].get('deterministic', True)),
        rect=False,
        multi_scale=False,
        baseline_disable_albumentations=True,
        val=False,
        cache='disk',
        patience=8,
    )

    leaderboard = []
    for cand in scaleaware_candidates:
        mode_name = cand['candidate_id']
        print(f'[SEARCH] training {mode_name} ...')
        run_dir = run_train_mode(
            mode=mode_name,
            config=pilot_cfg,
            mode_args=cand['ultralytics_args'],
            custom_augmentations=None,
        )
        score = _best_map5095_from_results_csv(run_dir / 'results.csv')
        leaderboard.append({
            'candidate_id': mode_name,
            'score_map50_95': score,
            'run_dir': run_dir.as_posix(),
            'ultralytics_args': cand['ultralytics_args'],
        })

    leaderboard = sorted(leaderboard, key=lambda x: x['score_map50_95'], reverse=True)
    dump_json(leaderboard, search_leaderboard_path)
else:
    leaderboard = load_json(search_leaderboard_path)

leaderboard_df = pd.DataFrame(leaderboard)
print('Top-5 candidates by pilot mAP50-95:')
print(leaderboard_df.head(5))

assert len(leaderboard) > 0, 'No scale-aware candidates were evaluated.'
best_scaleaware_candidate = leaderboard[0]
print('Best candidate:', best_scaleaware_candidate['candidate_id'])


In [ ]:
# Step 7: final training (Adaptive vs best Scale-Aware candidate).
from src.training.train_runner import TrainRunConfig, run_train_mode
from src.augmentation.policy_to_ultralytics import AugmentationPolicy
from src.utils.io import load_json

RUN_FINAL_TRAINING = False
FORCE_RERUN_FINAL = False

# ??????? ?????????? ?????????.
FINAL_PROFILE = 'hour_safe'  # fast | balanced | hour_safe | hour_full

profile_map = {
    'fast':      {'epochs': 8,  'imgsz': 960, 'batch': 8, 'fraction': 0.35, 'val': True, 'patience': 12},
    'balanced':  {'epochs': 18, 'imgsz': 960, 'batch': 8, 'fraction': 0.70, 'val': True, 'patience': 20},
    'hour_safe': {'epochs': 28, 'imgsz': 960, 'batch': 6, 'fraction': 1.00, 'val': True,  'patience': 30},
    'hour_full': {'epochs': 40, 'imgsz': 960, 'batch': 6, 'fraction': 1.00, 'val': True,  'patience': 45},
}
assert FINAL_PROFILE in profile_map, f'Unknown FINAL_PROFILE: {FINAL_PROFILE}'
prof = profile_map[FINAL_PROFILE]

final_cfg = TrainRunConfig(
    data_yaml=str(DATASET_ROOT / 'dataset.yaml'),
    model=str(runtime_cfg['training']['model']),
    epochs=int(prof['epochs']),
    imgsz=int(prof['imgsz']),
    batch=int(prof['batch']),
    device=0,
    workers=int(runtime_cfg['training'].get('workers', 2)),
    fraction=float(prof['fraction']),
    project_dir=str(RUNS_ROOT / 'final_compare'),
    seed=int(SEED),
    deterministic=bool(runtime_cfg['project'].get('deterministic', True)),
    rect=False,
    multi_scale=False,
    baseline_disable_albumentations=True,
    val=bool(prof['val']),
    cache='disk',
    patience=int(prof['patience']),
)

adaptive_policy_payload = load_json(adaptive_paths['policy_json'])
adaptive_policy = AugmentationPolicy(payload=adaptive_policy_payload)

adaptive_mode = 'adaptive_final'
scaleaware_mode = 'scaleaware_final'

adaptive_dir = RUNS_ROOT / 'final_compare' / adaptive_mode
scaleaware_dir = RUNS_ROOT / 'final_compare' / scaleaware_mode

if RUN_FINAL_TRAINING or FORCE_RERUN_FINAL or (not adaptive_dir.exists()) or (not scaleaware_dir.exists()):
    print('[FINAL] Training adaptive policy...')
    adaptive_run_dir = run_train_mode(
        mode=adaptive_mode,
        config=final_cfg,
        mode_args=adaptive_policy.get_ultralytics_train_args(),
        custom_augmentations=adaptive_policy.get_albumentations_transforms(seed=SEED),
    )

    print('[FINAL] Training scale-aware selected policy...')
    scaleaware_run_dir = run_train_mode(
        mode=scaleaware_mode,
        config=final_cfg,
        mode_args=best_scaleaware_candidate['ultralytics_args'],
        custom_augmentations=None,
    )
else:
    adaptive_run_dir = adaptive_dir
    scaleaware_run_dir = scaleaware_dir

print('adaptive_run_dir:', adaptive_run_dir)
print('scaleaware_run_dir:', scaleaware_run_dir)

assert (Path(adaptive_run_dir) / 'weights' / 'best.pt').exists(), 'Adaptive best.pt not found'
assert (Path(scaleaware_run_dir) / 'weights' / 'best.pt').exists(), 'Scale-aware best.pt not found'


In [ ]:
# Step 8: COCOeval for both final runs (AP_small, AP_tiny and others).
from pathlib import Path
from src.evaluation.predict_runner import predict_yolo_val_labels
from src.evaluation.coco_converter import convert_yolo_gt_to_coco, convert_yolo_pred_txt_to_coco
from src.evaluation.coco_eval_runner import run_coco_eval
from src.utils.io import dump_json

eval_dir = ARTIFACTS_ROOT / 'eval'
eval_dir.mkdir(parents=True, exist_ok=True)

class_names = runtime_cfg['dataset']['class_names']
image_extensions = tuple(runtime_cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp']))

val_images_dir = DATASET_ROOT / 'images' / 'val'
val_labels_dir = DATASET_ROOT / 'labels' / 'val'

coco_gt_path = eval_dir / 'coco_gt.json'
convert_yolo_gt_to_coco(
    images_dir=val_images_dir,
    labels_dir=val_labels_dir,
    class_names=class_names,
    output_path=coco_gt_path,
    image_extensions=image_extensions,
)

run_map = {
    'adaptive_rule_policy': Path(adaptive_run_dir),
    'scaleaware_autoaug': Path(scaleaware_run_dir),
}

metrics_by_method = {}
for method_name, run_dir in run_map.items():
    best_weights = run_dir / 'weights' / 'best.pt'

    pred_labels_dir = predict_yolo_val_labels(
        weights_path=best_weights,
        images_dir=val_images_dir,
        output_project=eval_dir / 'predict',
        run_name=method_name,
        imgsz=int(runtime_cfg['evaluation'].get('imgsz', prof['imgsz'])),
        device=0,
        conf=0.001,
        iou=0.6,
        max_det=500,
        use_tta=bool(runtime_cfg['evaluation'].get('use_tta', True)),
    )

    coco_dt_path = eval_dir / f'coco_dt_{method_name}.json'
    convert_yolo_pred_txt_to_coco(
        pred_labels_dir=pred_labels_dir,
        images_dir=val_images_dir,
        output_path=coco_dt_path,
        image_extensions=image_extensions,
    )

    metrics = run_coco_eval(
        coco_gt_path=coco_gt_path,
        coco_dt_path=coco_dt_path,
        output_path=eval_dir / f'coco_eval_{method_name}.json',
        use_tiny_eval=True,
        tiny_max_area=float(runtime_cfg['analysis']['tiny_max_area']),
    )
    metrics_by_method[method_name] = metrics

metrics_path = eval_dir / 'metrics_by_method.json'
dump_json(metrics_by_method, metrics_path)

print('Saved metrics:', metrics_path)
print(metrics_by_method)


In [ ]:
# Step 9: build final comparison table and save report artifacts.
import pandas as pd
from src.utils.io import dump_json

comparison_rows = []
for method_name, metrics in metrics_by_method.items():
    comparison_rows.append({
        'method': method_name,
        'AP@[0.5:0.95]': metrics.get('AP@[0.5:0.95]'),
        'AP50': metrics.get('AP50'),
        'AP_small': metrics.get('AP_small'),
        'AP_tiny': metrics.get('AP_tiny'),
        'AR_small': metrics.get('AR_small'),
        'AR_tiny': metrics.get('AR_tiny'),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(by='AP_small', ascending=False)
comparison_csv_path = ARTIFACTS_ROOT / 'comparison_table.csv'
comparison_md_path = ARTIFACTS_ROOT / 'comparison_table.md'
comparison_json_path = ARTIFACTS_ROOT / 'comparison_table.json'

comparison_df.to_csv(comparison_csv_path, index=False)
comparison_df.to_markdown(comparison_md_path, index=False)
dump_json(comparison_rows, comparison_json_path)

summary = {
    'best_method_by_AP_small': comparison_df.iloc[0]['method'] if len(comparison_df) else None,
    'profile': FINAL_PROFILE,
    'pilot_candidates': int(len(scaleaware_candidates)),
    'artifacts': {
        'comparison_csv': comparison_csv_path.as_posix(),
        'comparison_md': comparison_md_path.as_posix(),
        'comparison_json': comparison_json_path.as_posix(),
        'metrics_by_method': (ARTIFACTS_ROOT / 'eval' / 'metrics_by_method.json').as_posix(),
        'adaptive_policy_json': adaptive_paths['policy_json'],
        'scaleaware_candidates_json': (ARTIFACTS_ROOT / 'scaleaware_candidates' / 'scaleaware_candidates.json').as_posix(),
    }
}

dump_json(summary, ARTIFACTS_ROOT / 'summary.json')

print(comparison_df)
print('Summary saved:', ARTIFACTS_ROOT / 'summary.json')


## Notes

- ??? ???????? ?????? ?????????:
  - `RUN_SCALE_AWARE_SEARCH=True`
  - `RUN_FINAL_TRAINING=False`
- ??? ??????? ?????????:
  - `RUN_SCALE_AWARE_SEARCH=True`
  - `RUN_FINAL_TRAINING=True`
  - `FINAL_PROFILE='hour_safe'` ??? `'hour_full'`
- ??? ????????? ??????? ? `Google Drive`:
  - `.../experiments/small_objects_auto_aug/dota_scaleaware_vs_adaptive/`


In [ ]:
# Step X: unified monitoring export (train + val metrics history).
from pathlib import Path
import shutil
import pandas as pd

# This cell builds reusable CSV artifacts for later plots in thesis:
# 1) full epoch history across runs
# 2) last-epoch snapshot per run
# It includes both train-side metrics (losses) and val-side metrics (precision/recall/mAP).

PROJECT_ROOT_RESOLVED = Path(globals().get('PROJECT_ROOT', Path.cwd()))


def _resolve_runtime_cfg_path() -> Path | None:
    candidate_names = ['runtime_cfg_path']
    for name in candidate_names:
        value = globals().get(name)
        if value is not None:
            p = Path(value)
            if p.exists():
                return p

    hints = [
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'dota_runtime_config.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'xview_runtime_config.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'coco_small_runtime_config.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'dota_scaleaware_vs_adaptive_runtime.yaml',
        PROJECT_ROOT_RESOLVED / 'artifacts' / 'runtime_config.yaml',
    ]
    for p in hints:
        if p.exists():
            return p
    return None


def _load_cfg(cfg_path: Path | None):
    if cfg_path is None:
        return None
    try:
        import yaml
        return yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
    except Exception:
        return None


def _resolve_runs_root(cfg: dict | None) -> Path:
    name_candidates = [
        'RUNS_ROOT',
        'RUNS_DRIVE_DIR',
        'DOTA_EXPERIMENT_RUNS',
        'COCO_EXPERIMENT_RUNS',
        'XVIEW_EXPERIMENT_RUNS',
    ]
    for name in name_candidates:
        value = globals().get(name)
        if value is not None:
            p = Path(value)
            if p.exists():
                return p

    if isinstance(cfg, dict):
        p = cfg.get('training', {}).get('project_dir')
        if p:
            rp = Path(p)
            if rp.exists():
                return rp

    fallback = PROJECT_ROOT_RESOLVED / 'runs'
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback


def _resolve_drive_artifacts_root() -> Path | None:
    candidates = [
        globals().get('ARTIFACTS_ROOT'),
        globals().get('DOTA_EXPERIMENT_ARTIFACTS'),
        globals().get('COCO_EXPERIMENT_ARTIFACTS'),
        globals().get('XVIEW_EXPERIMENT_ARTIFACTS'),
        globals().get('ARTIFACTS_DRIVE_DIR'),
        globals().get('DRIVE_ARTIFACTS_ROOT'),
    ]
    for item in candidates:
        if item is None:
            continue
        p = Path(item)
        try:
            p.mkdir(parents=True, exist_ok=True)
            return p
        except Exception:
            continue
    return None


cfg_path = _resolve_runtime_cfg_path()
cfg = _load_cfg(cfg_path)
runs_root = _resolve_runs_root(cfg)

monitor_dir = PROJECT_ROOT_RESOLVED / 'artifacts' / 'monitoring'
monitor_dir.mkdir(parents=True, exist_ok=True)

results_files = sorted(runs_root.rglob('results.csv'))
if not results_files:
    raise FileNotFoundError(f'No results.csv found under runs root: {runs_root}')

frames = []
for csv_path in results_files:
    run_dir = csv_path.parent
    run_name = run_dir.name
    try:
        df = pd.read_csv(csv_path)
    except Exception as exc:
        print(f'[WARN] failed to read {csv_path}: {exc}')
        continue
    if df.empty:
        continue

    if 'epoch' not in df.columns:
        df['epoch'] = list(range(len(df)))

    df['run_name'] = run_name
    df['run_dir'] = run_dir.as_posix()
    frames.append(df)

if not frames:
    raise RuntimeError('No readable non-empty results.csv files were found.')

history_wide = pd.concat(frames, ignore_index=True)

# Keep numeric metric columns + id columns.
id_cols = ['run_name', 'run_dir', 'epoch']
metric_cols = [
    c for c in history_wide.columns
    if c not in id_cols and pd.api.types.is_numeric_dtype(history_wide[c])
]
history_wide = history_wide[id_cols + metric_cols]

history_wide_path = monitor_dir / 'metrics_history_wide.csv'
history_wide.to_csv(history_wide_path, index=False)

# Last-epoch snapshot per run (useful for quick comparison tables).
last_rows = history_wide.sort_values(['run_name', 'epoch']).groupby('run_name', as_index=False).tail(1)
last_rows = last_rows.sort_values('run_name').reset_index(drop=True)
last_rows_path = monitor_dir / 'metrics_last_epoch.csv'
last_rows.to_csv(last_rows_path, index=False)

# Build long-format table for plotting (run/epoch/metric/value).
history_long = history_wide.melt(
    id_vars=['run_name', 'run_dir', 'epoch'],
    value_vars=metric_cols,
    var_name='metric',
    value_name='value',
)

def _metric_split(metric_name: str) -> str:
    m = metric_name.lower()
    if m.startswith('metrics/') or m.startswith('val/'):
        return 'val'
    if m.startswith('train/') or m.endswith('_loss'):
        return 'train'
    if 'loss' in m:
        return 'train'
    return 'other'

history_long['split'] = history_long['metric'].map(_metric_split)
history_long_path = monitor_dir / 'metrics_history_long.csv'
history_long.to_csv(history_long_path, index=False)

# Quick console-friendly view for thesis workflow.
interesting = [
    'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
    'box_loss', 'cls_loss', 'dfl_loss',
    'metrics/precision(B)', 'metrics/recall(B)',
    'metrics/mAP50(B)', 'metrics/mAP50-95(B)',
]
show_cols = ['run_name', 'epoch'] + [c for c in interesting if c in last_rows.columns]
print('[OK] history_wide:', history_wide_path)
print('[OK] history_long:', history_long_path)
print('[OK] last_epoch:', last_rows_path)
print('\nLast-epoch snapshot (train + val):')
print(last_rows[show_cols].to_string(index=False))

# Optional Drive sync for artifacts.
drive_artifacts_root = _resolve_drive_artifacts_root()
if drive_artifacts_root is not None:
    drive_monitor_dir = drive_artifacts_root / 'monitoring'
    drive_monitor_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(history_wide_path, drive_monitor_dir / history_wide_path.name)
    shutil.copy2(history_long_path, drive_monitor_dir / history_long_path.name)
    shutil.copy2(last_rows_path, drive_monitor_dir / last_rows_path.name)
    print('[OK] Monitoring CSVs synced to Drive:', drive_monitor_dir)


In [ ]:
# Step Y: plot preview for train/val metrics (for thesis figures draft).
from pathlib import Path
import shutil
import pandas as pd
import matplotlib.pyplot as plt

RUN_PLOT_PREVIEW = True

if RUN_PLOT_PREVIEW:
    project_root = Path(globals().get('PROJECT_ROOT', Path.cwd()))
    monitor_dir = project_root / 'artifacts' / 'monitoring'
    history_long_path = monitor_dir / 'metrics_history_long.csv'

    if not history_long_path.exists():
        raise FileNotFoundError(
            f'Missing monitoring history: {history_long_path}. '
            'Run the monitoring export cell first.'
        )

    df = pd.read_csv(history_long_path)
    if df.empty:
        raise RuntimeError('metrics_history_long.csv is empty.')

    # Keep numeric rows only.
    df = df[pd.to_numeric(df['value'], errors='coerce').notna()].copy()
    df['value'] = df['value'].astype(float)

    plots_dir = monitor_dir / 'plots'
    plots_dir.mkdir(parents=True, exist_ok=True)

    # Metrics that are usually useful for diploma report.
    metric_candidates = [
        'metrics/mAP50-95(B)',
        'metrics/mAP50(B)',
        'metrics/precision(B)',
        'metrics/recall(B)',
        'train/box_loss',
        'train/cls_loss',
        'train/dfl_loss',
        'box_loss',
        'cls_loss',
        'dfl_loss',
    ]
    metrics = [m for m in metric_candidates if m in set(df['metric'].unique())]

    if not metrics:
        raise RuntimeError('No known metrics found for plotting in metrics_history_long.csv.')

    created = []

    # 1) Per-metric comparison across runs.
    for metric in metrics:
        sub = df[df['metric'] == metric].copy()
        if sub.empty:
            continue

        plt.figure(figsize=(9, 5))
        for run_name, grp in sub.groupby('run_name'):
            grp = grp.sort_values('epoch')
            plt.plot(grp['epoch'], grp['value'], marker='o', markersize=2, linewidth=1.6, label=str(run_name))

        plt.title(f'{metric} vs epoch')
        plt.xlabel('epoch')
        plt.ylabel(metric)
        plt.grid(alpha=0.3)
        plt.legend(loc='best', fontsize=8)
        out = plots_dir / f"metric_{metric.replace('/', '_').replace('(', '').replace(')', '')}.png"
        plt.tight_layout()
        plt.savefig(out, dpi=160)
        plt.close()
        created.append(out)

    # 2) Overview: best val metrics by run (last epoch).
    last_rows = (
        df.sort_values(['run_name', 'epoch'])
          .groupby(['run_name', 'metric'], as_index=False)
          .tail(1)
    )

    val_metrics_for_bar = [m for m in ['metrics/mAP50-95(B)', 'metrics/mAP50(B)', 'metrics/precision(B)', 'metrics/recall(B)'] if m in set(last_rows['metric'])]
    if val_metrics_for_bar:
        pivot = (
            last_rows[last_rows['metric'].isin(val_metrics_for_bar)]
            .pivot_table(index='run_name', columns='metric', values='value', aggfunc='mean')
            .sort_index()
        )

        ax = pivot.plot(kind='bar', figsize=(10, 5), rot=25)
        ax.set_title('Final val metrics by run (last epoch)')
        ax.set_ylabel('value')
        ax.grid(axis='y', alpha=0.3)
        out_bar = plots_dir / 'final_val_metrics_bar.png'
        plt.tight_layout()
        plt.savefig(out_bar, dpi=160)
        plt.close()
        created.append(out_bar)

    print('[OK] Plot files created:')
    for p in created:
        print(' -', p)

    # Optional Drive sync.
    drive_candidates = [
        globals().get('ARTIFACTS_ROOT'),
        globals().get('DOTA_EXPERIMENT_ARTIFACTS'),
        globals().get('COCO_EXPERIMENT_ARTIFACTS'),
        globals().get('XVIEW_EXPERIMENT_ARTIFACTS'),
        globals().get('ARTIFACTS_DRIVE_DIR'),
        globals().get('DRIVE_ARTIFACTS_ROOT'),
    ]
    drive_root = None
    for c in drive_candidates:
        if c is None:
            continue
        try:
            p = Path(c)
            p.mkdir(parents=True, exist_ok=True)
            drive_root = p
            break
        except Exception:
            continue

    if drive_root is not None:
        drive_plots_dir = drive_root / 'monitoring' / 'plots'
        drive_plots_dir.mkdir(parents=True, exist_ok=True)
        for p in created:
            shutil.copy2(p, drive_plots_dir / p.name)
        print('[OK] Plot files synced to Drive:', drive_plots_dir)
else:
    print('Plot preview skipped. Set RUN_PLOT_PREVIEW=True to enable.')
